
# *1.* Tokenization <br>

A Tokenization é a primeira fase na construção de uma arquitetura Transformers. <br> Consiste em transformar o dataset em palavras mais curtas (tokens) mas sem perder o seu significado.




In [2]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file ("/content/Os Lusiadas Tokenizer.json") #Tokenizer criado para este dataset, não é comum. Modeos atuais utilizam já tokenizers construídos.

with open ("/content/Os Lusiadas.txt", "r", encoding = "utf-8") as file:
  doc = file.read()

dados_tokenizer = tokenizer.encode (doc) #Encode encarrega-se de converter o dataset para Tokens. Todos os Tokens têm o seu ID

print (dados_tokenizer.tokens[:100])
print (dados_tokenizer.ids[:100])
print (len(dados_tokenizer.tokens))

['O', 'S', 'L', 'U', 'S', 'Í', 'A', 'D', 'A', 'S', 'Lu', 'ís', 'de', 'Cam', 'ões', 'Canto', 'I', 'As', 'armas', 'e', 'os', 'Bar', 'ões', 'assinala', 'dos', 'Que', 'da', 'Oci', 'dent', 'al', 'praia', 'Lusitana', 'Por', 'mares', 'nunca', 'de', 'antes', 'navega', 'dos', 'Passa', 'ram', 'ainda', 'além', 'da', 'Ta', 'probana', ',', 'Em', 'perigos', 'e', 'guerras', 'esforça', 'dos', 'Mais', 'do', 'que', 'prome', 'tia', 'a', 'força', 'humana', ',', 'E', 'entre', 'gente', 'remota', 'edifica', 'ram', 'Novo', 'Reino', ',', 'que', 'tanto', 'sublima', 'ram', ';', 'E', 'também', 'as', 'memórias', 'gloriosas', 'Daqueles', 'Reis', 'que', 'foram', 'dila', 'tando', 'A', 'Fé', ',', 'o', 'Império', ',', 'e', 'as', 'terras', 'vi', 'cios', 'as', 'De']
[27, 31, 24, 33, 31, 70, 14, 17, 14, 31, 3537, 465, 100, 1187, 340, 1823, 22, 239, 526, 43, 93, 2172, 340, 2680, 133, 125, 104, 2418, 2234, 110, 1571, 1221, 171, 919, 453, 100, 460, 1576, 133, 2386, 191, 3565, 4435, 104, 2189, 4670, 8, 312, 852, 43, 1655, 246

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as FUNCTION

tensor = torch.tensor(dados_tokenizer.ids, dtype = torch.long) #Colocação dos IDS num tensor de dimensão [1, 0]
print (tensor)
print (len(tensor))

#Divisão dos Dados
n = int (0.9 * len(tensor))
dados_treino = tensor [:n] #90% Dados de Treino
dados_teste = tensor [n:] #10% Dados de Teste

tensor([  27,   31,   24,  ...,  147, 2255,   10])
79565


### *1.2* Context Length e Batch Size <br>
É completamente impossível alimentar uma arquitetura Transformer com os dados logo todos de uma vez. Computacionalmente é impossível. <br> Para resolver isto, vamos alimentando o modelo com excertos de Tokens. <br>
**Context Lenght** é uma métrica que define quantos Tokens o modelo vê numa sequência. <br>
**Batch Size** é quantas sequências o modelo vê ao mesmo tempo.

In [5]:
context_len = 8
batch_size = 4

torch.manual_seed(1000)

def get_batch ():
  position = torch.randint (0, len(dados_treino) - context_len, (batch_size, ))
  x = torch.stack ([dados_treino [i : i + context_len] for i in position])
  y = torch.stack ([dados_treino [i + 1 : i + context_len + 1] for i in position])

  return x, y

inputs, targets = get_batch()

print ('inputs')
print (inputs.shape) #torch.Size([4, 8])
print (inputs)

print ('targets')
print (targets.shape) #torch.Size([4, 8])
print (targets)


inputs
torch.Size([4, 8])
tensor([[  95, 1810,  589, 2717,  210,   39,  520, 2861],
        [3832,  101, 1956, 2646,  191,   11,   63,   71],
        [4665,   12,  414, 3292,  104,  189,  100, 1956],
        [1638,  729,    8, 1982,  192, 1935,   43,  192]])
targets
torch.Size([4, 8])
tensor([[1810,  589, 2717,  210,   39,  520, 2861,   43],
        [ 101, 1956, 2646,  191,   11,   63,   71,  156],
        [  12,  414, 3292,  104,  189,  100, 1956,   39],
        [ 729,    8, 1982,  192, 1935,   43,  192, 1219]])


# *2.* Embedding

Embedding é um método para representar tokens como vetores num espaço contínuo.

Nos Transformers, o Embedding transforma tokens (IDs inteiros) em vetores numéricos que o modelo consegue processar.

Cada token passa a ser representado por um vetor de dimensão fixa, onde diferentes dimensões capturam diferentes características.

Por exemplo, palavras como "mãe" e "pai" tendem a ter representações semelhantes.



<b> Exemplo: </b> <br>

vocab = ["mãe", "pai", "avõ"] <br>

1Após a Tokenization: <br>

"mãe" -> 0 <br>
"pai" -> 1 <br>
"avõ" -> 2 <br>

O que o Embedding faz: <br>

0 -> [0.2, 0.3, 0.7] <br>
1 -> [0.5, 0.2, 0.3] <br>
2 -> [0.9, 0.5, 0.2] <br>

Onde entra a aprendizagem? <br>

Como os IDS são representados por um vetor de números, à medida que o modelo vai aprendendo, o mesmo vai ajustando esses valores e desta maneira, consegue aprender relações cada vez melhores.

In [29]:
number_embeddings = 32
vocab_size = tokenizer.get_vocab_size()

embedding = nn.Embedding (vocab_size, number_embeddings)

inputs_embedding = embedding (inputs)

print (inputs_embedding.shape, "\n")
print ("Primeira linha do tensor antes do Embedding:")
print (inputs [:1], "\n")

print ("Primeira linha do tensor após o Embedding")
print (inputs_embedding [:1], "\n")

print (f"ID {inputs [0,0]} pós Embedding --> {inputs_embedding [0,0]}")


torch.Size([4, 8, 32]) 

Primeira linha do tensor antes do Embedding:
tensor([[  95, 1810,  589, 2717,  210,   39,  520, 2861]]) 

Primeira linha do tensor após o Embedding
tensor([[[-7.4430e-01,  2.6509e-01, -9.6649e-01, -9.7557e-01, -1.3218e-02,
           1.0622e+00, -3.2592e-01,  1.5923e-01, -1.0577e-01,  1.6501e+00,
          -1.3806e+00, -1.1797e+00,  9.4188e-01, -7.6861e-01,  9.2887e-02,
          -4.4798e-01,  1.6737e+00, -6.6872e-02,  6.4115e-02, -1.7791e-01,
           4.2984e-01, -1.1478e+00,  2.8396e-01, -5.1180e-01, -1.2903e+00,
           2.1205e+00, -1.8935e-01,  7.7015e-01,  7.0498e-01,  1.0942e+00,
          -1.2421e+00, -1.8540e+00],
         [-1.1430e+00, -1.4554e+00,  1.0658e-01,  4.4421e-01, -1.0605e+00,
           2.2285e+00, -7.0684e-01,  3.9412e-01,  8.5464e-01,  9.0401e-01,
           9.1238e-01, -1.0402e+00,  7.8964e-01,  3.2547e-01,  9.9684e-02,
           5.4843e-01,  1.1926e+00,  1.5674e-01, -1.0889e-01, -3.4401e-01,
           1.2170e+00, -4.7727e-02,  7.8

# Positional Encoding

Positional Encoding é uma componente crucial arquiteturas Transformers. <br>
Ao contrário das RNNs, que processam os tokens sequencialmente, os Transformers processam todos os tokens em paralelo, tornando-os intrinsecamente alheios à ordem dos tokens. <br>
Positional Encoding resolve esta limitação ao incorporar informações posicionais nos dados de entrada.

<b> Exemplo: </b>

"O gato comeu o peixe" <br>
"O peixe comeu o gato" <br>

Para um Transformer isto seria:

["O", "gato", "comeu", "o", "peixe"] <br>
["O", "peixe", "comeu", "o", "gato"] <br>

Os Transformers veem tudo em paralelo, não têm noção de ordem por defeito. <br>
Após Embedding: <br>

"gato" -> ID 10 -> [0.2, 0.4] <br>
"comeu" -> ID 45 -> [0.2, 0.3] <br>
"peixe" -> ID 109 -> [0.3, 0,4] <br>

Positional Encoding: <br>

Posição 0 -> [0.2, 0.5] <br>
Posição 1 -> [0.4, 0.3] <br>
Posição 2 -> [0.7, 0.1] <br>

<b> O que o modelo tem que ver ? </b> <br>
Embedding + Positional Encoding

"gato" -> [0.2, 0.4] + [0.2, 0.5] <br>
"comeu" -> [0.2, 0.3] + [0.4, 0.3] <br>
"peixe" -> [0.3, 0,4] + [0.7, 0.1] <br>

<h2> Embedding -> “o que é a palavra” <br>
Positional Encoding -> “onde está a palavra” </h2>

In [31]:
positional_encoding = nn.Embedding (context_len, number_embeddings)

position = torch.arange (context_len)

inputs_final = inputs_embedding + positional_encoding (position)

print (inputs_final.shape)
print (inputs_embedding [0, 0])
print (inputs_final [0, 0])

torch.Size([4, 8, 32])
tensor([-0.7443,  0.2651, -0.9665, -0.9756, -0.0132,  1.0622, -0.3259,  0.1592,
        -0.1058,  1.6501, -1.3806, -1.1797,  0.9419, -0.7686,  0.0929, -0.4480,
         1.6737, -0.0669,  0.0641, -0.1779,  0.4298, -1.1478,  0.2840, -0.5118,
        -1.2903,  2.1205, -0.1893,  0.7701,  0.7050,  1.0942, -1.2421, -1.8540],
       grad_fn=<SelectBackward0>)
tensor([-4.2890,  1.3287, -0.7072, -0.0899, -0.4025, -1.7913, -1.0119,  0.4173,
         1.2554,  2.1386, -2.1108, -0.4315,  1.4290, -0.0970,  1.6097, -0.4724,
         2.7333,  1.6548,  0.8044, -1.2127,  0.7739,  0.9268,  0.3135, -0.0401,
        -1.2530,  0.7589, -0.9254,  0.2714,  0.0081,  1.2826, -0.8421, -1.6581],
       grad_fn=<SelectBackward0>)


# Attention
Attention responde a esta pergunta:

<b> “Para entender este token, a que outros tokens devo prestar atenção?” </b>

Para responder a essa pergunta, a Attention cria 3 coisas:

Query (Q) -> O que eu procuro ? <br>
Key (K) -> O que eu ofereço ? <br>
Value (V) -> A informação que eu passo <br>

Estas 3 métricas são valores aleatórios inicialmente, que depois são aprendidos pelo modelo. São aplicados token a token.



<b> Exemplo: </b>

"O gato bebeu o leite porque ele estava com sede"

Ele quem ? <br>
Attention responde a isso.

In [90]:
head_size = 8
number_head = 4 #32 / 8

''' Single Head Attention
Query = nn.Linear (number_embeddings, head_size, bias = False)
Key = nn.Linear (number_embeddings, head_size, bias = False)
Value = nn.Linear (number_embeddings, head_size, bias = False)

Q = Query (inputs_final) #y = inputs_final * Wq
K = Key (inputs_final) #y = inputs_final * Wk
V = Value (inputs_final) #y = inputs_final * Wv

print (f"B T C = {inputs_final.shape} \n")

print (f"B T C = {Q.shape}")
print (Q[0,0], "\n")

print (f"B T C = {K.shape}")
print (K[0,0], "\n")

print (f"B T C = {V.shape}")
print (V[0,0])

attention_scores = Q @ K.transpose (-2, -1) #@ = produto de vetores B T 8 @ B 8 T = B T T
tril = torch.tril(torch.ones(attention_scores.shape[-2], attention_scores.shape[-1]))
attention_scores = attention_scores.masked_fill (tril == 0, float('-inf'))
print (attention_scores.shape) #BTC -> 4,8,8
print (attention_scores[0,0])

attention_scores = torch.softmax (attention_scores, dim = -1) #Logits
print (attention_scores.shape)
print (attention_scores[0,0]) #Probabilidades

attention_scores = attention_scores @ V
#print (attention_scores.shape)
#print (attention_scores[0,0]) '''

B, T, C = inputs_final.shape #Batch | Sequence Length | Embeddings
#print (B, T, C)

QKV = nn.Linear (number_embeddings, number_embeddings * 3, bias = False)
QKV = QKV(inputs_final)

Q, K, V = QKV.split(C, dim = -1)
#print (Q.shape, K.shape, V.shape)

Q = Q.view (B, T, number_head, head_size)
K = K.view (B, T, number_head, head_size)
V = V.view (B, T, number_head, head_size)
#print (Q.shape, K.shape, V.shape)

Q = Q.transpose (1, 2)
K = K.transpose (1, 2)
V = V.transpose (1, 2)
#print (Q.shape, K.shape, V.shape)

attention_scores = Q @ K.transpose (-2, -1)
attention_scores = attention_scores / (head_size ** 0.5)
#print (attention_scores.shape)
tril = torch.tril(torch.ones(T, T, device = inputs_final.device))
tril = tril.unsqueeze(0).unsqueeze(0)
attention_scores = attention_scores.masked_fill (tril == 0, float('-inf'))
#print (attention_scores.shape)
#print (attention_scores[0,0])

attention_scores = FUNCTION.softmax (attention_scores, dim = -1) #Logits
#print (attention_scores.shape)
#print (attention_scores[0,0])

final = attention_scores @ V
#print (final.shape)
#print (final[0,0])

final = final.transpose(1, 2)
final = final.contiguous().view(B, T, number_embeddings)
#print (final.shape)
#print (final[0,0])

projection = nn.Linear (number_embeddings, number_embeddings)
final = projection (final)

final += inputs_final

layernorm = nn.LayerNorm (number_embeddings)
final = layernorm (final)